In [0]:
"""
01_bronze_ingestion.py

Purpose:
- Load raw CSV files from S3
- Convert all columns to STRING
- Store raw Delta bronze tables
"""

from pyspark.sql.functions import col

# --------------------------------------------------
# Create schema
# --------------------------------------------------
#catalog_name = "de_case_study"
#schema_name = "lakehouse"
#spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog_name}.{schema_name}")

#Newer Databricks Free Edition environments, you usually do not have permission to create a custom catalog so use default for assignment

In [0]:
# --------------------------------------------------
# File paths
# --------------------------------------------------

# Files are downloaded and saved in local from the path "https://merkle-de-interview-case-study.s3.eu-central-1.amazonaws.com/de/item.csv"
#and "https://merkle-de-interview-case-study.s3.eu-central-1.amazonaws.com/de/event.csv"

# --------------------------------------------------
# Read CSV files
# --------------------------------------------------

item_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", False)
     .csv("/Workspace/Users/khushbu675971@gmail.com/de-case-study/input_feeds/item.csv")
)

event_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", False)
    .option('escape', '"')
    .csv("/Workspace/Users/khushbu675971@gmail.com/de-case-study/input_feeds/event.csv")
)

In [0]:
# --------------------------------------------------
# Convert all columns to STRING
# --------------------------------------------------

item_df = item_df.select(
    [col(c).cast("string").alias(c) for c in item_df.columns]
)

for c in event_df.columns:
    clean_col = (
        c.replace(".", "_")
         .replace(" ", "_")
         .lower()
    )
    event_df = event_df.withColumnRenamed(c, clean_col)

event_df = event_df.select(
    [col(c).cast("string").alias(c) for c in event_df.columns]
)    
# --------------------------------------------------
# Save Bronze Tables
# --------------------------------------------------

item_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("bronze_item")

event_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("bronze_event")

print("Bronze layer completed successfully.")